# 🥈 Module 2: Silver Layer - Data Transformation

## Overview

In this notebook, we'll transform the raw Bronze data into a clean, validated Silver dataset using SQL.

**What you'll learn:**
- Apply data quality transformations with SQL
- Convert data types correctly
- Create Delta Lake tables
- Handle data validation

---

**Prerequisites:**
- Completed Module 1 (Bronze Layer)
- CSV files exist in `Files/bronze/trips/`

## Step 1: Load Bronze Data

First, let's load all the CSV files from the Bronze layer and create a unified view.

In [ ]:
# Load all CSV files from Bronze layer
bronze_path = "Files/bronze/trips/*.csv"

# Read all CSV files with headers
df_bronze = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv(bronze_path)

# Add source file information for lineage
from pyspark.sql.functions import input_file_name, regexp_extract

df_bronze = df_bronze.withColumn("_source_file", 
    regexp_extract(input_file_name(), r"([^/]+\.csv)$", 1))

# Create temporary view for SQL
df_bronze.createOrReplaceTempView("bronze_trips_raw")

print(f"✅ Loaded {df_bronze.count():,} records from Bronze layer")
print(f"   Source files: {df_bronze.select('_source_file').distinct().count()}")

In [ ]:
%%sql
-- Preview the raw bronze data
SELECT * FROM bronze_trips_raw LIMIT 5

In [ ]:
%%sql
-- Check current schema (all columns are strings from CSV)
DESCRIBE bronze_trips_raw

## Step 2: Analyze Data Quality Issues

Before transforming, let's understand what quality issues exist in our data.

In [ ]:
%%sql
-- Data quality analysis
SELECT 
    COUNT(*) as total_records,
    
    -- Null checks
    SUM(CASE WHEN started_at IS NULL OR started_at = '' THEN 1 ELSE 0 END) as null_started_at,
    SUM(CASE WHEN ended_at IS NULL OR ended_at = '' THEN 1 ELSE 0 END) as null_ended_at,
    SUM(CASE WHEN duration IS NULL OR duration = '' THEN 1 ELSE 0 END) as null_duration,
    SUM(CASE WHEN start_station_id IS NULL OR start_station_id = '' THEN 1 ELSE 0 END) as null_start_station,
    SUM(CASE WHEN end_station_id IS NULL OR end_station_id = '' THEN 1 ELSE 0 END) as null_end_station,
    
    -- Duration checks
    SUM(CASE WHEN CAST(duration AS INT) <= 0 THEN 1 ELSE 0 END) as invalid_duration,
    SUM(CASE WHEN CAST(duration AS INT) > 86400 THEN 1 ELSE 0 END) as duration_over_24h,
    
    -- Coordinate checks (Oslo approximate bounds: lat 59.8-60.0, lon 10.5-10.9)
    SUM(CASE WHEN CAST(start_station_latitude AS DOUBLE) < 59.5 
             OR CAST(start_station_latitude AS DOUBLE) > 60.2 THEN 1 ELSE 0 END) as invalid_start_lat
             
FROM bronze_trips_raw

In [ ]:
%%sql
-- Check duration distribution
SELECT 
    CASE 
        WHEN CAST(duration AS INT) <= 0 THEN '0: Invalid (<=0)'
        WHEN CAST(duration AS INT) <= 60 THEN '1: Under 1 min'
        WHEN CAST(duration AS INT) <= 300 THEN '2: 1-5 min'
        WHEN CAST(duration AS INT) <= 900 THEN '3: 5-15 min'
        WHEN CAST(duration AS INT) <= 1800 THEN '4: 15-30 min'
        WHEN CAST(duration AS INT) <= 3600 THEN '5: 30-60 min'
        WHEN CAST(duration AS INT) <= 7200 THEN '6: 1-2 hours'
        WHEN CAST(duration AS INT) <= 86400 THEN '7: 2-24 hours'
        ELSE '8: Over 24 hours'
    END as duration_bucket,
    COUNT(*) as trip_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bronze_trips_raw), 2) as percentage
FROM bronze_trips_raw
GROUP BY 1
ORDER BY 1

## Step 3: Create Silver Table with SQL Transformations

Now we'll create the Silver table with proper data types and quality flags.

**Transformations applied:**
1. Parse timestamps from strings
2. Cast duration to integer
3. Cast coordinates to double
4. Generate unique trip ID
5. Add data quality flags
6. Add metadata columns

In [ ]:
%%sql
-- Create or replace the Silver trips table
CREATE OR REPLACE TABLE silver_trips
USING DELTA
AS
SELECT 
    -- Generate unique trip ID using hash of key fields
    sha2(concat_ws('|', 
        started_at, 
        ended_at, 
        start_station_id, 
        end_station_id,
        duration
    ), 256) as trip_id,
    
    -- Timestamp conversions
    TO_TIMESTAMP(started_at) as started_at,
    TO_TIMESTAMP(ended_at) as ended_at,
    
    -- Duration as integer
    CAST(duration AS INT) as duration_seconds,
    
    -- Start station info
    start_station_id,
    start_station_name,
    start_station_description as start_station_desc,
    CAST(start_station_latitude AS DOUBLE) as start_latitude,
    CAST(start_station_longitude AS DOUBLE) as start_longitude,
    
    -- End station info
    end_station_id,
    end_station_name,
    end_station_description as end_station_desc,
    CAST(end_station_latitude AS DOUBLE) as end_latitude,
    CAST(end_station_longitude AS DOUBLE) as end_longitude,
    
    -- Metadata
    _source_file as source_file,
    CURRENT_DATE() as ingestion_date,
    
    -- Data quality validation
    CASE 
        WHEN started_at IS NULL OR started_at = '' THEN FALSE
        WHEN ended_at IS NULL OR ended_at = '' THEN FALSE
        WHEN CAST(duration AS INT) <= 0 THEN FALSE
        WHEN CAST(duration AS INT) > 86400 THEN FALSE  -- Over 24 hours
        WHEN start_station_id IS NULL OR start_station_id = '' THEN FALSE
        WHEN end_station_id IS NULL OR end_station_id = '' THEN FALSE
        ELSE TRUE
    END as is_valid,
    
    -- Quality issue description
    CASE 
        WHEN started_at IS NULL OR started_at = '' THEN 'Missing start timestamp'
        WHEN ended_at IS NULL OR ended_at = '' THEN 'Missing end timestamp'
        WHEN CAST(duration AS INT) <= 0 THEN 'Invalid duration (<=0)'
        WHEN CAST(duration AS INT) > 86400 THEN 'Duration over 24 hours'
        WHEN start_station_id IS NULL OR start_station_id = '' THEN 'Missing start station'
        WHEN end_station_id IS NULL OR end_station_id = '' THEN 'Missing end station'
        ELSE NULL
    END as quality_issues
    
FROM bronze_trips_raw

In [ ]:
%%sql
-- Verify the new schema
DESCRIBE silver_trips

## Step 4: Validate the Silver Table

In [ ]:
%%sql
-- Count total records and check quality distribution
SELECT 
    COUNT(*) as total_records,
    SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) as valid_records,
    SUM(CASE WHEN NOT is_valid THEN 1 ELSE 0 END) as invalid_records,
    ROUND(SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as valid_percentage
FROM silver_trips

In [ ]:
%%sql
-- Breakdown of quality issues
SELECT 
    COALESCE(quality_issues, 'No issues - Valid') as quality_status,
    COUNT(*) as record_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM silver_trips), 2) as percentage
FROM silver_trips
GROUP BY quality_issues
ORDER BY record_count DESC

In [ ]:
%%sql
-- Preview the cleaned data
SELECT 
    trip_id,
    started_at,
    ended_at,
    duration_seconds,
    start_station_name,
    end_station_name,
    is_valid
FROM silver_trips
WHERE is_valid = TRUE
LIMIT 10

## Step 5: Validate Data Types and Calculations

In [ ]:
%%sql
-- Verify timestamp calculations match duration
SELECT 
    started_at,
    ended_at,
    duration_seconds as stored_duration,
    TIMESTAMPDIFF(SECOND, started_at, ended_at) as calculated_duration,
    duration_seconds - TIMESTAMPDIFF(SECOND, started_at, ended_at) as difference
FROM silver_trips
WHERE is_valid = TRUE
LIMIT 10

In [ ]:
%%sql
-- Check unique stations (useful for dimension table planning)
SELECT 
    'Start Stations' as station_type,
    COUNT(DISTINCT start_station_id) as unique_count
FROM silver_trips
WHERE is_valid = TRUE

UNION ALL

SELECT 
    'End Stations' as station_type,
    COUNT(DISTINCT end_station_id) as unique_count
FROM silver_trips
WHERE is_valid = TRUE

In [ ]:
%%sql
-- Monthly distribution of trips (validates data completeness)
SELECT 
    DATE_FORMAT(started_at, 'yyyy-MM') as month,
    COUNT(*) as trip_count,
    SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) as valid_trips,
    ROUND(AVG(duration_seconds) / 60, 1) as avg_duration_minutes
FROM silver_trips
GROUP BY DATE_FORMAT(started_at, 'yyyy-MM')
ORDER BY month

## Step 6: Create Summary Statistics

In [ ]:
%%sql
-- Summary statistics for valid trips
SELECT 
    MIN(started_at) as first_trip,
    MAX(started_at) as last_trip,
    COUNT(*) as total_valid_trips,
    COUNT(DISTINCT start_station_id) as unique_stations,
    MIN(duration_seconds) as min_duration_sec,
    MAX(duration_seconds) as max_duration_sec,
    ROUND(AVG(duration_seconds), 0) as avg_duration_sec,
    ROUND(AVG(duration_seconds) / 60, 1) as avg_duration_min
FROM silver_trips
WHERE is_valid = TRUE

In [ ]:
%%sql
-- Top 10 most popular start stations
SELECT 
    start_station_id,
    start_station_name,
    start_latitude,
    start_longitude,
    COUNT(*) as trip_count
FROM silver_trips
WHERE is_valid = TRUE
GROUP BY start_station_id, start_station_name, start_latitude, start_longitude
ORDER BY trip_count DESC
LIMIT 10

## Step 7: Check Delta Lake Table Properties

In [ ]:
%%sql
-- View table history (Delta Lake versioning)
DESCRIBE HISTORY silver_trips

In [ ]:
%%sql
-- View detailed table properties
DESCRIBE DETAIL silver_trips

## ✅ Module Complete!

### Summary

In this module, you have:

1. ✅ Loaded raw CSV data from Bronze layer
2. ✅ Analyzed data quality issues
3. ✅ Created Silver table with proper data types
4. ✅ Applied data quality rules and flags
5. ✅ Validated the transformed data

### Key Transformations Applied

| Transformation | Description |
|---------------|-------------|
| Type Casting | Strings → Timestamps, Integers, Doubles |
| ID Generation | Created unique trip_id using SHA-256 hash |
| Quality Flags | is_valid boolean and quality_issues description |
| Metadata | Added source_file and ingestion_date |

### Key Takeaways

- **Data quality is measurable** - We quantified issues before fixing them
- **Delta Lake enables versioning** - Can time-travel to previous versions
- **Quality flags preserve data** - We kept invalid records but flagged them
- **SQL is powerful for transformations** - Complex logic in a single query

### Next Step

Continue to **Module 3: Gold Layer** where you'll:
- Design a dimensional model (Star Schema)
- Create Fact and Dimension tables
- Optimize for analytical queries

👉 Open `03-gold-dimensional-model.ipynb`